## IndicTrans2 Finetuning on Kaggle (Auto-Resume Enabled)

This notebook is specifically configured for **Kaggle's 2x T4 GPU environment** with a 12-hour session limit.

### How it works:
- **Data**: Streams directly from your Hugging Face dataset.
- **Checkpointing**: Every 500 steps, a model checkpoint is saved and pushed to Hugging Face.
- **Resuming**: If your 12-hour session expires, simply restart the notebook. Cell 4 will download your last checkpoint, and Cell 5 will seamlessly resume training from where it left off.

In [ ]:
# Cell 1: Install `uv` (Fast dependency manager)
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ['PATH']}"

In [ ]:
# Cell 2: Clone the translation pipeline repository
!git clone https://github.com/Firoj-Wise/Translation_en_indic_finetuning.git
%cd Translation_en_indic_finetuning
!uv sync

In [ ]:
# Cell 3: Authenticate with Weights & Biases and Hugging Face
# (It is recommended to store these in Kaggle Add-ons -> Secrets named WANDB_API_KEY and HF_TOKEN)
import wandb
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))
    login(token=user_secrets.get_secret("HF_TOKEN"))
except Exception as e:
    print("Secrets not found. Using hardcoded tokens...")
    # PASTE YOUR TOKENS HERE IF SECRETS FAIL:
    wandb.login(key="YOUR_WANDB_API_KEY_HERE")
    login(token="YOUR_HUGGING_FACE_TOKEN_HERE")

In [ ]:
# Cell 4: ♻️ Auto-Resume Preparation
# Download the last checkpoint from Hugging Face before starting, to resume if the session died earlier.

import os
os.makedirs("./checkpoints", exist_ok=True)
!uv run hf download Firoj112/indictrans2-en-npi-mai-finetuned --local-dir ./checkpoints

In [ ]:
# Cell 5: 🚀 Kick off Finetuning!
# This streams data, tokenizes, trains, evaluates, and pushes checkpoints + models to HF automatically.
# Note: Increased batch size to 32 for better GPU utilization on T4 x2.

!uv run accelerate launch --multi_gpu --num_processes=2 --num_machines=1 --dynamo_backend=no --mixed_precision=fp16 \
    run_pipeline.py --config configs/default.yaml \
    --override hub.push_to_hub=true \
               hub.model_id=Firoj112/indictrans2-en-npi-mai-finetuned \
               training.mixed_precision=fp16